# Face Recognition Pipeline — Open-Set, SCRFD detector, ArcFace + iResNet50



In [ ]:
print("test")

In [ ]:
import os
p = '/kaggle/input/datasets/yakhyokhuja/webface-112x112/webface_112x112'
print(os.listdir(p)[:5])

In [ ]:
# Cell 1: Environment check + installs
import torch, sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("No GPU detected — go to Runtime > Change runtime type > select GPU (T4), then rerun this cell")

In [ ]:
# Cell 2: Install onnxruntime-gpu + cuDNN9 + pin numpy/scipy/sklearn (hardened for Kaggle + Colab)
import shutil, os, glob, subprocess

site_packages = subprocess.run(
    ["python3", "-c", "import site; print(site.getsitepackages()[0])"],
    capture_output=True, text=True
).stdout.strip()
stale_dir = os.path.join(site_packages, "onnxruntime")
if os.path.exists(stale_dir):
    shutil.rmtree(stale_dir)
    print(f"Removed stale directory: {stale_dir}")

!pip uninstall -y onnxruntime onnxruntime-gpu -q

# Install insightface (and deps) BEFORE onnxruntime-gpu — insightface pulls in plain
# CPU-only onnxruntime as a transitive dep, which silently overwrites onnxruntime-gpu
# if installed afterward.
!pip install -q insightface opencv-python-headless scikit-learn
!pip install --no-cache-dir -q nvidia-cudnn-cu12==9.*
!pip uninstall -y onnxruntime -q
!pip install --no-cache-dir -q --force-reinstall onnxruntime-gpu==1.20.0

# Pin numpy + reinstall scipy/scikit-learn together against it — some environments
# (Kaggle's base image especially) ship a numpy version whose compiled ABI breaks
# scipy/scikit-learn's C extensions (ImportError: cannot import name '_center' etc.)
# Must reinstall all three together, not numpy alone, or scipy stays mismatched.
!pip install -q --force-reinstall "numpy<2.3" scipy scikit-learn

print("\nInstall complete. RESTART THE RUNTIME NOW (Restart & Clear Cell Outputs on Kaggle,")
print("or Runtime > Restart session on Colab — NOT Factory Reset, which wipes installs),")
print("then continue with Cell 3 below — do not re-run Cell 1/2 first.")

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login
login(token="YOUR_HF_TOKEN_HERE")

In [ ]:
!pip install -q --force-reinstall "numpy<2.3"
print("numpy pinned.")

In [ ]:
# Cell 3: Post-restart verification — run this FIRST after restarting, before anything else
import os, glob

cudnn_libs = glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/cudnn/lib")
if cudnn_libs:
    os.environ["LD_LIBRARY_PATH"] = cudnn_libs[0] + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print("LD_LIBRARY_PATH set to:", cudnn_libs[0])
else:
    print("WARNING: cuDNN lib path not found — GPU execution provider will likely fail")

import onnxruntime as ort
print("ORT version:", ort.__version__)
print("Available providers:", ort.get_available_providers())

assert "CUDAExecutionProvider" in ort.get_available_providers(), \
    "CUDAExecutionProvider missing — do not proceed, re-run Cell 2 and restart again"
print("\nGPU execution provider confirmed working.")

In [ ]:
!pip install -q --force-reinstall "numpy<2.3" scipy scikit-learn
print("numpy + scipy + sklearn reinstalled together.")

In [1]:
# Cell 4: Confirm library versions
import cv2, numpy as np, onnxruntime, sklearn, insightface

print("OpenCV:", cv2.__version__)
print("ONNX Runtime:", onnxruntime.__version__)
print("scikit-learn:", sklearn.__version__)
print("InsightFace:", insightface.__version__)

OpenCV: 4.13.0
ONNX Runtime: 1.20.0
scikit-learn: 1.9.0
InsightFace: 1.0.1


In [2]:
# Cell 5: Load SCRFD detector — restricted to detection module only (saves VRAM/load time vs full buffalo_l pack)
from insightface.app import FaceAnalysis

app = FaceAnalysis(name='buffalo_l', allowed_modules=['detection'],
                    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

print("Detector loaded. Confirm above that 'Applied providers' shows CUDAExecutionProvider first.")

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:02<00:00, 97180.74KB/s] 


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvid

In [3]:
# Cell 6: Face alignment — 5-point landmarks -> 112x112 crop (ArcFace reference template)
import numpy as np
import cv2

ARCFACE_DST = np.array([
    [38.2946, 51.6963],
    [73.5318, 51.5014],
    [56.0252, 71.7366],
    [41.5493, 92.3655],
    [70.7299, 92.2041]
], dtype=np.float32)

def align_face(img, kps, image_size=112):
    tform = cv2.estimateAffinePartial2D(kps, ARCFACE_DST, method=cv2.LMEDS)[0]
    aligned = cv2.warpAffine(img, tform, (image_size, image_size), borderValue=0.0)
    return aligned

def get_embedding(img_bgr, model, detector_app, device):
    """Detect -> align -> embed. Returns None if no face detected (caller must handle)."""
    faces = detector_app.get(img_bgr)
    if len(faces) == 0:
        return None
    aligned = align_face(img_bgr, faces[0].kps)
    tensor = torch.from_numpy(aligned).permute(2, 0, 1).float().unsqueeze(0).to(device)
    tensor = (tensor - 127.5) / 128.0
    with torch.no_grad():
        emb = model(tensor)
    return emb.cpu().numpy().flatten()

app_small = FaceAnalysis(name='buffalo_l', allowed_modules=['detection'],
                          providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app_small.prepare(ctx_id=0, det_size=(160, 160))

print("Alignment function + two detector configs ready: `app` (det_size=640, full-scene images),")
print("`app_small` (det_size=160, small pre-cropped images like LFW).")

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
model ignore: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvid

In [4]:
# Cell 7: iResNet backbone — no final ReLU, ends in BatchNorm1d, unrestricted 512-D output
import torch
import torch.nn as nn

class IRBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.bn0 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.prelu = nn.PReLU(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = x
        out = self.bn0(x)
        out = self.conv1(out)
        out = self.bn1(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        return out + identity

class IResNet(nn.Module):
    def __init__(self, embedding_dim=512, layers=(3, 4, 14, 3), channels=(64, 128, 256, 512)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 3, 1, 1, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.PReLU(channels[0]),
        )
        blocks = []
        in_ch = channels[0]
        for n_layers, out_ch in zip(layers, channels):
            for i in range(n_layers):
                stride = 2 if i == 0 else 1
                blocks.append(IRBlock(in_ch, out_ch, stride))
                in_ch = out_ch
        self.body = nn.Sequential(*blocks)
        self.bn_out = nn.BatchNorm2d(channels[-1])
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(channels[-1] * 7 * 7, embedding_dim)
        self.bn_fc = nn.BatchNorm1d(embedding_dim)

    def forward(self, x):
        x = self.stem(x)
        x = self.body(x)
        x = self.bn_out(x)
        x = self.dropout(x)
        x = x.flatten(1)
        x = self.fc(x)
        x = self.bn_fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_test_model = IResNet(embedding_dim=512, layers=(3,4,14,3)).to(device)
_x = torch.randn(4, 3, 112, 112).to(device)
_out = _test_model(_x)
print("Output shape:", _out.shape)
print("Contains negative values (expected True — unrestricted output):", (_out < 0).any().item())
print("Params (M):", sum(p.numel() for p in _test_model.parameters())/1e6)
del _test_model, _x, _out

Output shape: torch.Size([4, 512])
Contains negative values (expected True — unrestricted output): True
Params (M): 43.590848


In [5]:
# Cell 8: ArcMarginProduct — normalizes weights+features, adds angular margin
import math
import torch.nn.functional as F

class ArcMarginProduct(nn.Module):
    def __init__(self, embedding_dim, num_classes, margin=0.5, scale=64.0):
        super().__init__()
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, embeddings, labels):
        emb_norm = F.normalize(embeddings, p=2, dim=1)
        W_norm = F.normalize(self.weight, p=2, dim=1)
        cosine = F.linear(emb_norm, W_norm).clamp(-1 + 1e-7, 1 - 1e-7)
        sine = torch.sqrt(1.0 - cosine.pow(2))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        logits = (one_hot * phi + (1.0 - one_hot) * cosine) * self.scale
        return logits

print("ArcMarginProduct defined.")

ArcMarginProduct defined.


In [6]:
print(device)

cuda


In [7]:
# Cell 9: Dummy overfit sanity check — verifies backprop is wired correctly before real training
import torch.optim as optim

torch.manual_seed(0)
num_identities = 20
mock_images = torch.randn(num_identities, 3, 112, 112).to(device)
mock_labels = torch.arange(num_identities).to(device)

_model = IResNet(embedding_dim=512, layers=(3,4,14,3)).to(device)
_head = ArcMarginProduct(embedding_dim=512, num_classes=num_identities, margin=0.5, scale=64.0).to(device)
_optimizer = optim.SGD(list(_model.parameters()) + list(_head.parameters()), lr=0.1, momentum=0.9)
_loss_fn = nn.CrossEntropyLoss()

_model.train(); _head.train()
for epoch in range(30):
    embeddings = _model(mock_images)
    logits = _head(embeddings, mock_labels)
    loss = _loss_fn(logits, mock_labels)
    _optimizer.zero_grad()
    loss.backward()
    _optimizer.step()
    acc = (logits.argmax(1) == mock_labels).float().mean().item()
    if epoch % 5 == 0 or epoch == 29:
        print(f"epoch {epoch:2d}  loss={loss.item():.4f}  acc={acc*100:.1f}%")

print("\nPASS" if loss.item() < 0.05 and acc == 1.0 else "\nFAIL — check architecture/backprop before proceeding")
del _model, _head, _optimizer, _loss_fn

epoch  0  loss=37.2624  acc=0.0%
epoch  5  loss=10.5937  acc=25.0%
epoch 10  loss=0.9070  acc=90.0%
epoch 15  loss=0.1294  acc=95.0%
epoch 20  loss=0.0140  acc=100.0%
epoch 25  loss=0.0065  acc=100.0%
epoch 29  loss=0.0180  acc=100.0%

PASS


In [8]:
# Cell 10: Load LFW verification pairs (via sklearn's maintained mirror — avoids flaky UMass server)
from sklearn.datasets import fetch_lfw_pairs

lfw_pairs = fetch_lfw_pairs(subset='test', color=True, resize=1.0)

print("Data shape:", lfw_pairs.pairs.shape)
print("Num pairs:", len(lfw_pairs.target))
print("Genuine pairs:", (lfw_pairs.target == 1).sum())
print("Impostor pairs:", (lfw_pairs.target == 0).sum())

Data shape: (1000, 2, 125, 94, 3)
Num pairs: 1000
Genuine pairs: 500
Impostor pairs: 500


In [9]:
# Cell 11: Extract embeddings for all LFW pairs (sanity check on untrained model)
import numpy as np
from tqdm import tqdm

def to_uint8_bgr(img_float):
    img = (img_float * 255).astype(np.uint8)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

_sanity_model = IResNet(embedding_dim=512, layers=(3,4,14,3)).to(device)
_sanity_model.eval()

embeddings_a, embeddings_b, valid_labels = [], [], []
skipped = 0

for i in tqdm(range(len(lfw_pairs.target))):
    img_a = to_uint8_bgr(lfw_pairs.pairs[i, 0])
    img_b = to_uint8_bgr(lfw_pairs.pairs[i, 1])

    emb_a = get_embedding(img_a, _sanity_model, app_small, device)
    emb_b = get_embedding(img_b, _sanity_model, app_small, device)

    if emb_a is None or emb_b is None:
        skipped += 1
        continue

    embeddings_a.append(emb_a)
    embeddings_b.append(emb_b)
    valid_labels.append(lfw_pairs.target[i])

print(f"\nProcessed {len(valid_labels)} pairs, skipped {skipped} (detection failures)")
embeddings_a = np.array(embeddings_a)
embeddings_b = np.array(embeddings_b)
valid_labels = np.array(valid_labels)
print("Embeddings shape:", embeddings_a.shape)
del _sanity_model

100%|██████████| 1000/1000 [00:23<00:00, 42.53it/s]


Processed 1000 pairs, skipped 0 (detection failures)
Embeddings shape: (1000, 512)


In [10]:
# Cell 12: Proper LFW 10-fold verification protocol (standard benchmark methodology)
from sklearn.model_selection import KFold

def l2_normalize(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def lfw_10fold_eval(embeddings_a, embeddings_b, labels, n_folds=10):
    emb_a_norm = l2_normalize(embeddings_a)
    emb_b_norm = l2_normalize(embeddings_b)
    cosine_sims = np.sum(emb_a_norm * emb_b_norm, axis=1)

    kf = KFold(n_splits=n_folds, shuffle=False)
    fold_accs = []
    thresholds = np.arange(-1.0, 1.0, 0.005)

    indices = np.arange(len(labels))
    for train_idx, test_idx in kf.split(indices):
        best_acc, best_thresh = 0, 0
        for t in thresholds:
            preds = (cosine_sims[train_idx] > t).astype(int)
            acc = (preds == labels[train_idx]).mean()
            if acc > best_acc:
                best_acc, best_thresh = acc, t
        test_preds = (cosine_sims[test_idx] > best_thresh).astype(int)
        test_acc = (test_preds == labels[test_idx]).mean()
        fold_accs.append(test_acc)

    fold_accs = np.array(fold_accs)
    return fold_accs.mean(), fold_accs.std(), cosine_sims

mean_acc, std_acc, cosine_sims = lfw_10fold_eval(embeddings_a, embeddings_b, valid_labels)
print(f"LFW 10-fold accuracy: {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%")
print("(Untrained model — expect ~55-65%. This is a weak positive bias from raw pixel")
print(" statistics an untrained CNN still picks up, not evidence of a bug.)")
print("Realistic target after full training: ~99.5-99.8%. 100% is not achievable on LFW —")
print("published SOTA (ArcFace, iResNet100, MS1M-scale) tops out ~99.83%; LFW itself has")
print("some mislabeled pairs, and human performance is estimated around 97.5%.")

LFW 10-fold accuracy: 49.60% +/- 15.78%
(Untrained model — expect ~55-65%. This is a weak positive bias from raw pixel
 statistics an untrained CNN still picks up, not evidence of a bug.)
Realistic target after full training: ~99.5-99.8%. 100% is not achievable on LFW —
published SOTA (ArcFace, iResNet100, MS1M-scale) tops out ~99.83%; LFW itself has
some mislabeled pairs, and human performance is estimated around 97.5%.


In [11]:
# # Cell 13: Get CASIA-WebFace — download zip to Drive once, unzip locally each session (fast)
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# DRIVE_DIR = "/content/drive/MyDrive/face_recognition_data"
# LOCAL_DIR = "/content/data"
# os.makedirs(DRIVE_DIR, exist_ok=True)
# os.makedirs(LOCAL_DIR, exist_ok=True)

# zip_path = f"{DRIVE_DIR}/webface-112x112.zip"

# if not os.path.exists(zip_path):
#     !pip install -q kaggle
#     from google.colab import files
#     print("Upload your kaggle.json")
#     uploaded = files.upload()
#     os.makedirs("/root/.kaggle", exist_ok=True)
#     !cp kaggle.json /root/.kaggle/
#     !chmod 600 /root/.kaggle/kaggle.json
#     !kaggle datasets download -d yakhyokhuja/webface-112x112 -p {DRIVE_DIR}
#     print("Downloaded zip to Drive (persists across sessions).")
# else:
#     print("Zip already in Drive — skipping download.")

# # Unzip locally (fast) every session, since /content is wiped on runtime reset
# if not os.path.exists(f"{LOCAL_DIR}/webface_112x112"):
#     !unzip -q {zip_path} -d {LOCAL_DIR}
#     print("Unzipped to local disk.")
# else:
#     print("Already unzipped locally this session.")

# print("\n--- Verifying ---")
# !du -sh {LOCAL_DIR}
# !find {LOCAL_DIR} -maxdepth 2 -type d | head -5
# !find {LOCAL_DIR} -iname "*.jpg" | wc -l

In [12]:
# Cell 14: Build train/val manifest — OPEN-SET split (val identities never seen in training)
import os, csv, random
from pathlib import Path
from collections import defaultdict

root = Path("/kaggle/input/datasets/yakhyokhuja/webface-112x112/webface_112x112")# confirm this matches Cell 13's folder output; adjust if different

def scan_identities(root, min_images_per_id=5):
    identity_to_paths = defaultdict(list)
    identity_dirs = [d for d in root.iterdir() if d.is_dir()]
    print(f"Found {len(identity_dirs)} identity folders")
    for identity_dir in sorted(identity_dirs):
        imgs = [p for p in identity_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
        if len(imgs) >= min_images_per_id:
            identity_to_paths[identity_dir.name] = imgs
    print(f"Kept {len(identity_to_paths)} identities with >= {min_images_per_id} images")
    return identity_to_paths

def split_identities(identity_to_paths, val_frac=0.02, min_val_images=8, seed=42):
    rng = random.Random(seed)
    ids = list(identity_to_paths.keys())
    rng.shuffle(ids)
    val_candidates = [i for i in ids if len(identity_to_paths[i]) >= min_val_images]
    n_val = max(1, int(len(ids) * val_frac))
    val_ids = set(rng.sample(val_candidates, min(n_val, len(val_candidates))))
    train_ids = [i for i in ids if i not in val_ids]
    print(f"Split: {len(train_ids)} train identities / {len(val_ids)} val identities (disjoint — open-set)")
    return train_ids, list(val_ids)

def write_manifest(identity_to_paths, identity_list, out_csv, label_map=None):
    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["image_path", "identity", "label"])
        for identity in identity_list:
            label = label_map[identity] if label_map else identity
            for p in identity_to_paths[identity]:
                writer.writerow([str(p), identity, label])
    print(f"Wrote {out_csv}")

identity_to_paths = scan_identities(root, min_images_per_id=5)
train_ids, val_ids = split_identities(identity_to_paths, val_frac=0.02)
train_label_map = {identity: idx for idx, identity in enumerate(sorted(train_ids))}

os.makedirs("/content/prepared", exist_ok=True)
write_manifest(identity_to_paths, train_ids, "/content/prepared/train_manifest.csv", train_label_map)
write_manifest(identity_to_paths, val_ids, "/content/prepared/val_manifest.csv", label_map=None)

with open("/content/prepared/num_train_classes.txt", "w") as f:
    f.write(str(len(train_label_map)))

print(f"\nTrain classes: {len(train_label_map)}")

Found 10572 identity folders
Kept 10569 identities with >= 5 images
Split: 10358 train identities / 211 val identities (disjoint — open-set)
Wrote /content/prepared/train_manifest.csv
Wrote /content/prepared/val_manifest.csv

Train classes: 10358


In [13]:
# Cell 15: Dataset class + DataLoader
import csv
from collections import defaultdict
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

class FaceManifestDataset(Dataset):
    def __init__(self, manifest_csv, transform=None):
        self.samples = []
        with open(manifest_csv) as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.samples.append((row["image_path"], int(row["label"])))
        self.transform = transform
        self.label_to_indices = defaultdict(list)
        for idx, (_, label) in enumerate(self.samples):
            self.label_to_indices[label].append(idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = FaceManifestDataset("/content/prepared/train_manifest.csv", transform=train_transform)
print("Total training images:", len(train_ds))
print("Total identities:", len(train_ds.label_to_indices))

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, drop_last=True, pin_memory=True)

batch_imgs, batch_labels = next(iter(train_loader))
print("\nBatch images shape:", batch_imgs.shape)
print("Batch labels shape:", batch_labels.shape)
print("Label range in this batch:", batch_labels.min().item(), "-", batch_labels.max().item())
print("Pixel value range:", batch_imgs.min().item(), "to", batch_imgs.max().item())

Total training images: 481618
Total identities: 10358

Batch images shape: torch.Size([128, 3, 112, 112])
Batch labels shape: torch.Size([128])
Label range in this batch: 15 - 10059
Pixel value range: -1.0 to 1.0


In [17]:
# Cell 16: Training loop — SGD + linear warmup + step LR decay + gradient clipping
import torch
import torch.nn as nn
import torch.optim as optim
from huggingface_hub import HfApi, upload_file, list_repo_files, hf_hub_download
import time
import os

# NOTE: Check this path! The comment says Kaggle-local, but the string still says /content/
# You might need to change it to something like "/kaggle/working/prepared/num_train_classes.txt"
num_classes = int(open("/content/prepared/num_train_classes.txt").read().strip())

model = IResNet(embedding_dim=512, layers=(3,4,14,3)).to(device)
head = ArcMarginProduct(embedding_dim=512, num_classes=num_classes, margin=0.5, scale=64.0).to(device)

# ==========================================
# NEW: Wrap models in DataParallel for Multi-GPU
# ==========================================
if torch.cuda.device_count() > 1:
    print(f"Utilizing {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
    head = nn.DataParallel(head)
# ==========================================

base_lr = 0.025
optimizer = optim.SGD(list(model.parameters()) + list(head.parameters()),
                       lr=base_lr, momentum=0.9, weight_decay=5e-4)

warmup_steps = 500
def lr_lambda(step):
    return min(step / warmup_steps, 1.0)

warmup_scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
step_scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[8, 14, 20], gamma=0.1)
loss_fn = nn.CrossEntropyLoss()

HF_REPO = "zahir9897/face-recognition-checkpoints"
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
global_step = 0

def train_one_epoch(epoch, log_every=50):
    global global_step
    model.train(); head.train()
    running_loss, running_correct, running_total = 0.0, 0, 0
    t0 = time.time()

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        embeddings = model(images)
        logits = head(embeddings, labels)
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(model.parameters()) + list(head.parameters()), max_norm=5.0)
        optimizer.step()

        if global_step < warmup_steps:
            warmup_scheduler.step()
        global_step += 1

        running_loss += loss.item()
        running_correct += (logits.argmax(1) == labels).sum().item()
        running_total += labels.size(0)

        if i % log_every == 0:
            acc = running_correct / max(running_total, 1)
            elapsed = time.time() - t0
            imgs_per_sec = running_total / elapsed if elapsed > 0 else 0
            print(f"epoch {epoch} step {i}/{len(train_loader)}  loss={loss.item():.4f}  "
                  f"acc={acc*100:.2f}%  lr={optimizer.param_groups[0]['lr']:.5f}  {imgs_per_sec:.0f} img/s")

    step_scheduler.step()
    print(f"\n=== Epoch {epoch} done in {(time.time()-t0)/60:.1f} min, avg_loss={running_loss/len(train_loader):.4f} ===\n")

    # ==========================================
    # NEW: Safely extract state_dict whether using 1 GPU or Multiple GPUs
    # ==========================================
    model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    head_state = head.module.state_dict() if isinstance(head, nn.DataParallel) else head.state_dict()

    local_path = f"{CKPT_DIR}/ckpt_epoch{epoch}.pt"
    torch.save({"model": model_state, "head": head_state,
                "optimizer": optimizer.state_dict(), "epoch": epoch,
                "global_step": global_step}, local_path)
    upload_file(path_or_fileobj=local_path, path_in_repo=f"ckpt_epoch{epoch}.pt",
                repo_id=HF_REPO, repo_type="model")
    print(f"Saved and pushed ckpt_epoch{epoch}.pt to Hugging Face Hub")

Utilizing 2 GPUs!


In [20]:
# Cell 17: Resume-aware epoch runner — auto-loads latest checkpoint from Hugging Face Hub
def load_latest_checkpoint():
    global global_step
    try:
        files = [f for f in list_repo_files(HF_REPO) if f.startswith("ckpt_epoch")]
    except Exception:
        files = []
    if not files:
        print("No checkpoint found on HF Hub — starting from epoch 0")
        return 0
    latest_file = sorted(files, key=lambda f: int(f.split("epoch")[-1].split(".pt")[0]))[-1]
    local_path = hf_hub_download(repo_id=HF_REPO, filename=latest_file)
    ckpt = torch.load(local_path, map_location=device)
    
    # ==========================================
    # NEW: Safely load state_dict into DataParallel or single model
    # ==========================================
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(ckpt["model"])
        head.module.load_state_dict(ckpt["head"])
    else:
        model.load_state_dict(ckpt["model"])
        head.load_state_dict(ckpt["head"])
    # ==========================================
    
    optimizer.load_state_dict(ckpt["optimizer"])
    global_step = ckpt.get("global_step", 0)
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from {latest_file}, starting at epoch {start_epoch}")
    return start_epoch

start_epoch = load_latest_checkpoint()
N_EPOCHS_THIS_RUN = 2  # keep small on Kaggle for now — safer than 30 given session instability today
for epoch in range(start_epoch, start_epoch + N_EPOCHS_THIS_RUN):
    train_one_epoch(epoch, log_every=50)

ckpt_epoch7.pt:   0%|          | 0.00/392M [00:00<?, ?B/s]

Resumed from ckpt_epoch7.pt, starting at epoch 8
epoch 8 step 0/3762  loss=13.9915  acc=19.53%  lr=0.02500  90 img/s
epoch 8 step 50/3762  loss=13.4924  acc=23.33%  lr=0.02500  189 img/s
epoch 8 step 100/3762  loss=12.7247  acc=23.28%  lr=0.02500  194 img/s
epoch 8 step 150/3762  loss=11.5348  acc=23.36%  lr=0.02500  197 img/s
epoch 8 step 200/3762  loss=11.8396  acc=23.12%  lr=0.02500  197 img/s
epoch 8 step 250/3762  loss=14.0213  acc=22.99%  lr=0.02500  198 img/s
epoch 8 step 300/3762  loss=13.2719  acc=22.81%  lr=0.02500  198 img/s
epoch 8 step 350/3762  loss=13.1131  acc=22.81%  lr=0.02500  198 img/s
epoch 8 step 400/3762  loss=11.3340  acc=22.70%  lr=0.02500  199 img/s
epoch 8 step 450/3762  loss=14.1147  acc=22.63%  lr=0.02500  199 img/s
epoch 8 step 500/3762  loss=13.7637  acc=22.51%  lr=0.02500  199 img/s
epoch 8 step 550/3762  loss=12.3033  acc=22.50%  lr=0.02500  199 img/s
epoch 8 step 600/3762  loss=14.0606  acc=22.45%  lr=0.02500  199 img/s
epoch 8 step 650/3762  loss=12.6

KeyboardInterrupt: 

In [21]:
# Create a NEW CELL and run this to force the learning rate and run Epochs 8 & 9
for param_group in optimizer.param_groups:
    param_group['lr'] = 0.00250

print(f"Active learning rate updated to: {optimizer.param_groups[0]['lr']:.5f}")

# Run Epoch 8 and 9 with the reduced learning rate
N_EPOCHS_THIS_RUN = 2  
for epoch in range(8, 8 + N_EPOCHS_THIS_RUN):
    train_one_epoch(epoch, log_every=50)

Active learning rate updated to: 0.00250
epoch 8 step 0/3762  loss=13.3782  acc=17.97%  lr=0.00250  89 img/s
epoch 8 step 50/3762  loss=11.3797  acc=26.61%  lr=0.00250  190 img/s
epoch 8 step 100/3762  loss=12.1366  acc=27.85%  lr=0.00250  195 img/s
epoch 8 step 150/3762  loss=11.0354  acc=28.70%  lr=0.00250  197 img/s
epoch 8 step 200/3762  loss=10.6511  acc=29.48%  lr=0.00250  197 img/s
epoch 8 step 250/3762  loss=10.8161  acc=30.13%  lr=0.00250  198 img/s
epoch 8 step 300/3762  loss=10.6803  acc=30.45%  lr=0.00250  198 img/s
epoch 8 step 350/3762  loss=8.1877  acc=30.91%  lr=0.00250  199 img/s
epoch 8 step 400/3762  loss=12.0740  acc=31.24%  lr=0.00250  199 img/s
epoch 8 step 450/3762  loss=9.7469  acc=31.60%  lr=0.00250  199 img/s
epoch 8 step 500/3762  loss=9.6921  acc=31.98%  lr=0.00250  199 img/s
epoch 8 step 550/3762  loss=11.2839  acc=32.24%  lr=0.00250  199 img/s
epoch 8 step 600/3762  loss=10.7770  acc=32.43%  lr=0.00250  199 img/s
epoch 8 step 650/3762  loss=8.8679  acc=32.

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved and pushed ckpt_epoch8.pt to Hugging Face Hub
epoch 9 step 0/3762  loss=8.2396  acc=46.09%  lr=0.00250  85 img/s
epoch 9 step 50/3762  loss=8.5704  acc=44.33%  lr=0.00250  186 img/s
epoch 9 step 100/3762  loss=7.9156  acc=44.61%  lr=0.00250  192 img/s
epoch 9 step 150/3762  loss=8.3335  acc=44.39%  lr=0.00250  196 img/s
epoch 9 step 200/3762  loss=7.9485  acc=44.65%  lr=0.00250  196 img/s
epoch 9 step 250/3762  loss=7.3374  acc=44.73%  lr=0.00250  197 img/s
epoch 9 step 300/3762  loss=8.7292  acc=44.75%  lr=0.00250  198 img/s
epoch 9 step 350/3762  loss=7.4017  acc=44.88%  lr=0.00250  198 img/s
epoch 9 step 400/3762  loss=6.9392  acc=44.82%  lr=0.00250  198 img/s
epoch 9 step 450/3762  loss=8.3715  acc=44.86%  lr=0.00250  198 img/s
epoch 9 step 500/3762  loss=7.9527  acc=44.86%  lr=0.00250  198 img/s
epoch 9 step 550/3762  loss=10.0183  acc=44.91%  lr=0.00250  198 img/s
epoch 9 step 600/3762  loss=8.8300  acc=44.86%  lr=0.00250  198 img/s
epoch 9 step 650/3762  loss=8.6341  acc=4

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved and pushed ckpt_epoch9.pt to Hugging Face Hub


In [ ]:
 # Cell 17: Overnight Training Runner — Interactive Session
def load_latest_checkpoint():
    global global_step
    try:
        files = [f for f in list_repo_files(HF_REPO) if f.startswith("ckpt_epoch")]
    except Exception:
        files = []
    if not files:
        print("No checkpoint found on HF Hub — starting from epoch 0")
        return 0
    latest_file = sorted(files, key=lambda f: int(f.split("epoch")[-1].split(".pt")[0]))[-1]
    local_path = hf_hub_download(repo_id=HF_REPO, filename=latest_file)
    ckpt = torch.load(local_path, map_location=device)
    
    # Safely load state_dict into DataParallel or single model
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(ckpt["model"])
        head.module.load_state_dict(ckpt["head"])
    else:
        model.load_state_dict(ckpt["model"])
        head.load_state_dict(ckpt["head"])
        
    optimizer.load_state_dict(ckpt["optimizer"])
    global_step = ckpt.get("global_step", 0)
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from {latest_file}, starting at epoch {start_epoch}")
    return start_epoch

# 1. Load latest checkpoint
start_epoch = load_latest_checkpoint()

# 2. Synchronize Learning Rate with milestones [8, 14, 20]
if start_epoch >= 20:
    target_lr = base_lr * (0.1 ** 3)
elif start_epoch >= 14:
    target_lr = base_lr * (0.1 ** 2)
elif start_epoch >= 8:
    target_lr = base_lr * 0.1
else:
    target_lr = base_lr

for param_group in optimizer.param_groups:
    param_group['lr'] = target_lr

step_scheduler.last_epoch = start_epoch
print(f"Active learning rate set to: {optimizer.param_groups[0]['lr']:.5f}")

# 3. Run training loop for the rest of the night
# 15 epochs at ~40 mins each = ~10 hours of training
N_EPOCHS_THIS_RUN = 15  
for epoch in range(start_epoch, start_epoch + N_EPOCHS_THIS_RUN):
    train_one_epoch(epoch, log_every=50)

ckpt_epoch9.pt:   0%|          | 0.00/392M [00:00<?, ?B/s]

Resumed from ckpt_epoch9.pt, starting at epoch 10
Active learning rate set to: 0.00250
epoch 10 step 0/3762  loss=5.9129  acc=43.75%  lr=0.00250  76 img/s
epoch 10 step 50/3762  loss=6.7831  acc=49.65%  lr=0.00250  206 img/s
epoch 10 step 100/3762  loss=7.9480  acc=49.74%  lr=0.00250  204 img/s
epoch 10 step 150/3762  loss=7.4608  acc=49.86%  lr=0.00250  205 img/s
epoch 10 step 200/3762  loss=7.5426  acc=49.80%  lr=0.00250  203 img/s
epoch 10 step 250/3762  loss=7.6037  acc=49.91%  lr=0.00250  202 img/s
epoch 10 step 300/3762  loss=8.1735  acc=49.85%  lr=0.00250  202 img/s
epoch 10 step 350/3762  loss=5.8184  acc=49.95%  lr=0.00250  202 img/s
epoch 10 step 400/3762  loss=7.3439  acc=49.88%  lr=0.00250  202 img/s
epoch 10 step 450/3762  loss=9.2310  acc=49.78%  lr=0.00250  201 img/s
epoch 10 step 500/3762  loss=8.1461  acc=49.76%  lr=0.00250  201 img/s
epoch 10 step 550/3762  loss=6.8157  acc=49.62%  lr=0.00250  201 img/s
epoch 10 step 600/3762  loss=8.3384  acc=49.57%  lr=0.00250  201 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved and pushed ckpt_epoch10.pt to Hugging Face Hub
epoch 11 step 0/3762  loss=7.3328  acc=52.34%  lr=0.00250  75 img/s
epoch 11 step 50/3762  loss=5.9200  acc=52.63%  lr=0.00250  189 img/s
epoch 11 step 100/3762  loss=7.8028  acc=53.45%  lr=0.00250  193 img/s
epoch 11 step 150/3762  loss=5.6307  acc=53.21%  lr=0.00250  196 img/s
epoch 11 step 200/3762  loss=8.7020  acc=53.14%  lr=0.00250  196 img/s
epoch 11 step 250/3762  loss=6.2349  acc=52.95%  lr=0.00250  197 img/s
epoch 11 step 300/3762  loss=6.7299  acc=53.01%  lr=0.00250  198 img/s
epoch 11 step 350/3762  loss=5.6977  acc=52.95%  lr=0.00250  198 img/s
epoch 11 step 400/3762  loss=7.0979  acc=52.95%  lr=0.00250  198 img/s
epoch 11 step 450/3762  loss=7.6557  acc=52.81%  lr=0.00250  198 img/s
epoch 11 step 500/3762  loss=6.8676  acc=52.78%  lr=0.00250  198 img/s
epoch 11 step 550/3762  loss=5.2030  acc=52.85%  lr=0.00250  198 img/s
epoch 11 step 600/3762  loss=6.2694  acc=52.83%  lr=0.00250  198 img/s
epoch 11 step 650/3762  loss

...

New Data Upload: |          |  0.00B /  0.00B            

Saved and pushed ckpt_epoch12.pt to Hugging Face Hub
epoch 13 step 0/3762  loss=5.2212  acc=63.28%  lr=0.00250  60 img/s
epoch 13 step 50/3762  loss=6.7161  acc=57.28%  lr=0.00250  187 img/s
epoch 13 step 100/3762  loss=4.5975  acc=57.05%  lr=0.00250  192 img/s
epoch 13 step 150/3762  loss=6.7722  acc=57.08%  lr=0.00250  196 img/s
epoch 13 step 200/3762  loss=5.5580  acc=57.18%  lr=0.00250  196 img/s
epoch 13 step 250/3762  loss=5.9348  acc=57.31%  lr=0.00250  197 img/s
epoch 13 step 300/3762  loss=5.7255  acc=57.38%  lr=0.00250  197 img/s
epoch 13 step 350/3762  loss=6.4806  acc=57.35%  lr=0.00250  198 img/s
epoch 13 step 400/3762  loss=6.4674  acc=57.27%  lr=0.00250  198 img/s
epoch 13 step 450/3762  loss=5.2960  acc=57.31%  lr=0.00250  198 img/s
epoch 13 step 500/3762  loss=4.4907  acc=57.20%  lr=0.00250  198 img/s
epoch 13 step 550/3762  loss=3.9914  acc=57.16%  lr=0.00250  198 img/s
epoch 13 step 600/3762  loss=4.9378  acc=57.13%  lr=0.00250  198 img/s
epoch 13 step 650/3762  loss

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved and pushed ckpt_epoch13.pt to Hugging Face Hub
epoch 14 step 0/3762  loss=7.0957  acc=55.47%  lr=0.00025  39 img/s
epoch 14 step 50/3762  loss=5.0855  acc=59.67%  lr=0.00025  162 img/s
epoch 14 step 100/3762  loss=4.6353  acc=60.23%  lr=0.00025  178 img/s
epoch 14 step 150/3762  loss=5.6759  acc=60.25%  lr=0.00025  186 img/s
epoch 14 step 200/3762  loss=4.2631  acc=60.68%  lr=0.00025  188 img/s
epoch 14 step 250/3762  loss=6.6275  acc=60.69%  lr=0.00025  191 img/s
epoch 14 step 300/3762  loss=6.9557  acc=60.64%  lr=0.00025  192 img/s
epoch 14 step 350/3762  loss=4.0762  acc=60.62%  lr=0.00025  193 img/s
epoch 14 step 400/3762  loss=4.2882  acc=60.74%  lr=0.00025  194 img/s
epoch 14 step 450/3762  loss=5.4914  acc=60.76%  lr=0.00025  194 img/s
epoch 14 step 500/3762  loss=5.6159  acc=60.87%  lr=0.00025  195 img/s
epoch 14 step 550/3762  loss=3.7613  acc=60.96%  lr=0.00025  195 img/s
epoch 14 step 600/3762  loss=5.5249  acc=61.01%  lr=0.00025  196 img/s
epoch 14 step 650/3762  loss

In [23]:
# Cell 18: Evaluate current checkpoint on LFW (10-fold protocol)
import numpy as np
import torch
from tqdm import tqdm

print("Starting LFW Evaluation on current weights...")
model.eval()
embeddings_a, embeddings_b, valid_labels = [], [], []
skipped = 0

# Added torch.no_grad() to speed up inference and prevent memory leaks
with torch.no_grad():
    for i in tqdm(range(len(lfw_pairs.target))):
        img_a = to_uint8_bgr(lfw_pairs.pairs[i, 0])
        img_b = to_uint8_bgr(lfw_pairs.pairs[i, 1])
        
        # get_embedding should work seamlessly with the DataParallel model
        emb_a = get_embedding(img_a, model, app_small, device)
        emb_b = get_embedding(img_b, model, app_small, device)
        
        if emb_a is None or emb_b is None:
            skipped += 1
            continue
            
        embeddings_a.append(emb_a)
        embeddings_b.append(emb_b)
        valid_labels.append(lfw_pairs.target[i])

embeddings_a = np.array(embeddings_a)
embeddings_b = np.array(embeddings_b)
valid_labels = np.array(valid_labels)
print(f"Processed {len(valid_labels)} pairs, skipped {skipped}")

mean_acc, std_acc, _ = lfw_10fold_eval(embeddings_a, embeddings_b, valid_labels)
print(f"\nLFW 10-fold accuracy: {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%")

# Put the model back into training mode for the next epochs
model.train()

Starting LFW Evaluation on current weights...


 84%|████████▎ | 835/1000 [00:34<00:06, 24.34it/s]


KeyboardInterrupt: 